# Rewrite the uncompacted timeseries tables

One-off compaction and sort pass over the three tables whose existing data has never been sorted:
`fcst_joined_timeseries` (52 GB), `sim_joined_timeseries` (14.7 GB) and `secondary_timeseries` (54 GB).

The nightly `routine_table_maintenance` flow cannot get these into shape on its own:

- The joined tables were skipped entirely until the binpack fallback landed — `get_rewrite_settings`
  ignored any table without a `write.sort-order` or `write.target-file-size-bytes` property.
- `secondary_timeseries` was never skipped, but at **1.21 files per partition** with only 14 partitions
  holding 5 or more files, `rewrite_data_files` (default `min-input-files=5`) passes over essentially the
  whole table. It needs an explicit `rewrite-all` once.

After this pass the nightly flow only ever sees incremental work.

**Why not just let the nightly job do it?** The `rewrite_data_files` task is `timeout_seconds = 45 * 60`
with `retries = 1`. A ~120 GB first pass will not finish inside that window, and a timeout would retry
from scratch and time out again. Every call below sets `partial-progress.enabled` so work commits in
batches and a timeout keeps whatever completed.

## Prerequisites, per table

| table | needs first | why |
| --- | --- | --- |
| `fcst_joined_timeseries` | nothing | partitioning and sort order already set by the Prefect workflow |
| `sim_joined_timeseries` | `03_preprocessing/simulation/01_create_joined_timeseries.ipynb` | that notebook sets `partition_by` and `write_ordered_by`; `rewrite_data_files` cannot change a partition spec |
| `secondary_timeseries` | migration `0011` via `00_apply_migrations.ipynb` | declares the real sort order; `strategy => 'sort'` fails without one |

Each section guards its own prerequisite and fails fast rather than doing the wrong work.

`primary_timeseries` is deliberately **not** included. Its partitioning is still an open question
(`months(value_time)` vs `years(value_time)`), and rewriting 19.7 GB before that is decided means paying
for it twice. `secondary_timeseries` is included because its spec is settled — the high-volume forecast
configurations already sit at 156-636 MB monthly.

You must have write permissions to the data warehouse. Expect this to run for a while — it is a full
rewrite of roughly 120 GB.

In [ ]:
import time

from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=20,
    executor_memory="32g",
    executor_cores=6
)

In [ ]:
# rewrite_data_files runs at most this many file groups concurrently. The
# default is 5 and an earlier run of this notebook used 10, which left the
# cluster almost idle: secondary_timeseries averages 25.6 MB per partition, so
# a file group is roughly one task, and 10 concurrent groups kept ~10 of 120
# cores busy. Set this near the core count for small groups.
MAX_CONCURRENT_GROUPS = 100
MAX_COMMITS = 50


def rewrite(table_name, where=None):
    """Sort-rewrite a table, optionally scoped to a subset of partitions.

    rewrite-all because the data has never been sorted: without it, groups
    already at target size are left alone and stay unsorted, and at ~1.2 files
    per partition that is nearly every partition.

    partial-progress commits in batches, so stopping partway keeps the work
    that finished. Nothing is lost if this is interrupted - uncommitted output
    files simply become orphans for the nightly flow to clean up.
    """
    scope = f", where => \"{where}\"" if where else ""
    return spark.sql(f"""
        CALL iceberg.system.rewrite_data_files(
            table => 'teehr.{table_name}',
            strategy => 'sort'{scope},
            options => map(
                'rewrite-all', 'true',
                'partial-progress.enabled', 'true',
                'partial-progress.max-commits', '{MAX_COMMITS}',
                'max-concurrent-file-group-rewrites', '{MAX_CONCURRENT_GROUPS}'
            )
        )
    """).collect()[0]

## Before

In [ ]:
def report(table_name):
    """Print file and partition layout so the rewrite can be measured."""
    f = spark.sql(f"""
        SELECT count(*) AS files,
               round(sum(file_size_in_bytes) / 1e9, 2) AS gb,
               round(avg(file_size_in_bytes) / 1e6, 1) AS avg_mb,
               round(avg(cardinality(split_offsets)), 2) AS row_groups_per_file
        FROM iceberg.teehr.{table_name}.files
    """).collect()[0]
    props = {
        r["key"]: r["value"]
        for r in spark.sql(f"SHOW TBLPROPERTIES iceberg.teehr.{table_name}").collect()
    }
    print(table_name)
    print(f"  {f['files']} files, {f['gb']} GB, {f['avg_mb']} MB avg, "
          f"{f['row_groups_per_file']} row groups/file")
    # Reading .partitions can fail on a table that has been through a partition
    # spec change (ValidationException: Cannot find source column for partition
    # field). secondary_timeseries has two specs, from migrations 0002 and 0006.
    # This is only a diagnostic, so never let it stop the rewrite.
    try:
        p = spark.sql(f"""
            SELECT count(*) AS partitions,
                   round(avg(file_count), 1) AS files_per_partition,
                   max(file_count) AS max_files
            FROM iceberg.teehr.{table_name}.partitions
        """).collect()[0]
        print(f"  {p['partitions']} partitions, {p['files_per_partition']} files/partition "
              f"(max {p['max_files']})")
    except Exception as exc:
        print(f"  partitions: unavailable ({type(exc).__name__})")
    # `sort-order` is Iceberg's reflection of a real sort order. A `write.sort-order`
    # key is the inert custom property from migrations 0002/0006 and means nothing.
    print(f"  sort order: {props.get('sort-order', '(none declared)')}")


for table in ["fcst_joined_timeseries", "sim_joined_timeseries", "secondary_timeseries"]:
    report(table)

## fcst_joined_timeseries

Partitioned by `months(value_time), configuration_name` and sorted by
`primary_location_id, secondary_location_id, reference_time, value_time` — both set by
`prefect-workflows/workflows/metrics/utils/joined_forecast_utils.py`. Nothing to change here; it just
needs compacting.

No `sort_order` argument is passed: `strategy => 'sort'` uses the table's declared sort order. Passing one
explicitly (as `03_rewrite_timeseries.ipynb` does) overrides it and can silently sort by the wrong
columns.

`rewrite-all` is set because the data has never been sorted. Without it, partitions holding fewer than
`min-input-files` (5) files would be left untouched and stay unsorted. Drop it on later runs.

In [ ]:
%%time
r = rewrite("fcst_joined_timeseries")
print(f"rewrote {r['rewritten_data_files_count']} files into "
      f"{r['added_data_files_count']}, failed={r['failed_data_files_count']}")

## sim_joined_timeseries

Requires `01_create_joined_timeseries.ipynb` to have been re-run first — see the note at the top. The
guard below fails fast rather than compacting 14.7 GB into the wrong layout.

In [ ]:
props = {
    r["key"]: r["value"]
    for r in spark.sql("SHOW TBLPROPERTIES iceberg.teehr.sim_joined_timeseries").collect()
}
if "sort-order" not in props:
    raise RuntimeError(
        "sim_joined_timeseries has no declared sort order. Re-run "
        "warehouse/remote/03_preprocessing/simulation/01_create_joined_timeseries.ipynb "
        "first - it sets partition_by and write_ordered_by. Rewriting now would compact "
        "the table into its current single unpartitioned spec."
    )
print(f"sort order declared: {props['sort-order']}")

In [ ]:
%%time
r = rewrite("sim_joined_timeseries")
print(f"rewrote {r['rewritten_data_files_count']} files into "
      f"{r['added_data_files_count']}, failed={r['failed_data_files_count']}")

## secondary_timeseries

Requires migration `0011` to have been applied — it declares
`location_id, reference_time, value_time` as the table's sort order. The legacy `write.sort-order`
property from migration 0006 is inert and does not count; `strategy => 'sort'` reads the real one and
fails with `Cannot sort data without a valid sort order` if it is missing.

Partitioning is unchanged. `months(value_time), configuration_name` is right for this table: the
high-volume forecast configurations already produce 156-636 MB monthly partitions, and coarsening to
`years()` would push them to 2.4-5.4 GB while discarding the `value_time` pruning that currently cuts a
`reference_time` lookup down to 3 splits.

Baseline to compare against, measured before the rewrite — one gage over a 1-day `reference_time`
window scanned 127,597,773 rows to return 960 (90.2 MB physical, 1.75 s).

This is the largest of the three at 54 GB.

### Run it in chunks

54 GB in one call is a long uninterruptible-feeling stretch, and `rewrite_data_files` has no resume
memory — re-running with `rewrite-all` redoes completed groups. So this is chunked by
`configuration_name`, largest first. Stop between chunks freely; add the finished names to `DONE` before
re-running so they are not redone.

The top five configurations are roughly 38 GB of the 54 GB and carry nearly all the query volume. The
long tail of `psu_*` and regional runs sits at sub-MB partitions and gains little — reasonable to skip.

In [ ]:
props = {
    r["key"]: r["value"]
    for r in spark.sql("SHOW TBLPROPERTIES iceberg.teehr.secondary_timeseries").collect()
}
if "sort-order" not in props:
    raise RuntimeError(
        "secondary_timeseries has no declared sort order. Apply migration 0011 via "
        "warehouse/remote/04_maintenance/00_apply_migrations.ipynb first. The "
        "write.sort-order property from migration 0006 is an inert custom key and "
        "is not what rewrite_data_files reads."
    )
print(f"sort order declared: {props['sort-order']}")

In [ ]:
# Derived from the table so it stays correct as configurations come and go.
rows = spark.sql("""
    SELECT partition.configuration_name AS cfg,
           round(sum(total_data_file_size_in_bytes) / 1e9, 2) AS gb
    FROM iceberg.teehr.secondary_timeseries.partitions
    GROUP BY 1 ORDER BY 2 DESC
""").collect()
# Files written before migration 0006 added configuration_name to the partition
# spec carry NULL for it here. Chunking by configuration would silently skip
# them and report success, so refuse to chunk if any exist.
CONFIGS = [(r["cfg"], r["gb"]) for r in rows if r["cfg"] is not None]
# Key this on the NULL group existing at all, not on its size - a small one
# rounds to 0.0 GB and would slip through.
unscoped = [r for r in rows if r["cfg"] is None]
if unscoped:
    raise RuntimeError(
        f"{unscoped[0]['gb']} GB is in partitions with no configuration_name, from "
        "before migration 0006 added it to the spec. Chunking would skip that data. "
        "Rewrite the table in one unscoped call instead: rewrite('secondary_timeseries')."
    )
if not CONFIGS:
    raise RuntimeError("no configurations found in secondary_timeseries.partitions")
print(f"{len(CONFIGS)} configurations, {sum(g for _, g in CONFIGS):.1f} GB total")

# Seed this by hand to resume after a kernel restart. Within a session the
# loop also remembers what it finished, so re-running this cell continues
# rather than starting over - `DONE = set()` on its own would wipe that.
ALREADY_DONE = set()
DONE = globals().get("DONE", set()) | ALREADY_DONE

# Trim this to stop after the configurations that matter; the tail is tiny.
TOP_N = len(CONFIGS)

for cfg, gb in CONFIGS[:TOP_N]:
    if cfg in DONE:
        print(f"skip {cfg} ({gb} GB) - already done")
        continue
    print(f"--- {cfg} ({gb} GB)", flush=True)
    started = time.time()
    r = rewrite("secondary_timeseries", where=f"configuration_name = '{cfg}'")
    print(f"    {r['rewritten_data_files_count']} -> {r['added_data_files_count']} files, "
          f"failed={r['failed_data_files_count']}, {time.time() - started:.0f}s", flush=True)
    DONE.add(cfg)

## After

Expect far fewer, larger files. Snapshot and orphan-file cleanup is left to the nightly
`routine_table_maintenance` flow, which expires snapshots older than 7 days — the pre-rewrite files stay
on S3 until then.

In [ ]:
for table in ["fcst_joined_timeseries", "sim_joined_timeseries", "secondary_timeseries"]:
    report(table)

In [ ]:
spark.stop()